# The Tokyo Medical University Scandal, as a Bias Case Study

In this notebook I look at the 2018 Tokyo Medical University entrance exam scandal, not as a news story, but as a case study in how a scoring system can produce a large, systematic gap in outcomes between two groups who started out equally capable.

The scandal itself was not caused by a machine learning model, it was people adjusting exam scores by hand. I am covering it here anyway because it keeps showing up in AI ethics discussions, and once I worked through why, the mechanism turned out to be exactly the kind of thing that can also hide inside an automated scoring or ranking system, which is what this notebook is really about.

In this notebook, I will:

- Summarize what was reported about the scandal
- Build a small synthetic simulation of a group-level score adjustment
- Measure the resulting gap with a standard disparate-impact test
- Show a subtler trap: a model can reproduce this same gap even after the protected attribute is removed, if it is trained on the biased outcomes
- Write down what I take from this for any AI system I build or use

This stays small and synthetic on purpose. It is not a re-analysis of the real dataset, which I do not have access to, just a way to make the mechanism concrete enough to reason about.

## 1. Background: What Was Reported

In 2018, Tokyo Medical University admitted that it had been adjusting entrance exam scores for years in a way that reduced the number of women admitted, and that also disadvantaged male applicants who were sitting the exam again after previously failing it, while favoring first-time male applicants.

This came to light during an unrelated investigation into the university, and once it did, several other Japanese medical schools acknowledged similar practices in their own admissions. Reporting at the time described the reasoning given internally as a kind of workforce planning: an assumption that women were more likely to reduce their working hours or leave clinical practice after marriage or childbirth, so admitting fewer of them was framed as easing a doctor shortage.

I am summarizing this from public reporting rather than the university's own data, which was never released in a form I could analyze, so the simulation later in this notebook is illustrative, not a reconstruction of the real numbers.

## 2. Why a Manual Scandal Belongs in an AI Ethics Notebook

No model was involved here, so it would be reasonable to ask why this belongs in a notebook about building AI agents. Three things about it map directly onto problems that show up in automated scoring systems:

- A single adjustment was applied to everyone in a group, rather than any individual being judged on their own record.
- The justification was a prediction about the group's future behavior, not anything about the specific applicant in front of the committee.
- The adjustment was invisible to the people it affected, because the scoring process itself was opaque, so nobody outside the university could check it.

All three of these can happen quietly inside a trained model too, a group-level pattern baked into weights instead of a spreadsheet formula, which is exactly why this case gets used as a teaching example well beyond Japan or medicine.

## 3. A Note on the Numbers Below

Everything from here on is a small synthetic simulation I built myself, using made-up scores from a random number generator, not the university's actual data. I am using it to make the *mechanism* concrete: what happens to outcomes when a blanket, group-level adjustment is applied to a scoring process, even when the two groups start out equally capable by construction.

None of the specific numbers below should be read as a claim about the real scandal's actual figures.

## 4. Simulating Two Equally Able Applicant Groups

I start by generating raw exam scores for two groups, drawn from the exact same distribution. Building them from the same distribution on purpose means any gap I see later in this notebook is coming from something I did to the scores, not from the groups actually differing in ability.

In [ ]:
import random
import statistics

random.seed(42)


def simulate_scores(count, mean=70, spread=10):
    """Returns a list of made-up exam scores drawn from the same distribution."""
    return [round(random.gauss(mean, spread), 1) for _ in range(count)]


group_a_scores = simulate_scores(500)
group_b_scores = simulate_scores(500)